# BAGEL Dataset Distribution-Distance Experiment (v2 — updated registry)

This checks whether the **5 new OOD test sets** are actually distributionally different from
the **9 current training datasets** — updated from the original 9-train/3-OOD version to match
the composition in `bagel_router_v2_distilbert.ipynb`:

- **`guychuk` dropped from training** (was dataset 9)
- **`nvidia/Aegis-AI-Content-Safety-Dataset-2.0` moved from OOD into training** (now `model_10`)
- **`ahsanayub/malicious-prompts` dropped from OOD** (disqualified — leaked Harelix rows +
  HackAPrompt attack-outcome labels rather than content-maliciousness labels)
- **5 new OOD test datasets** replace the old 3-dataset OOD pool

**Method, unchanged from before:** embed every dataset's prompts into a shared vector space
with a sentence-transformer, then compare distributions pairwise (Gaussian KL, Jensen-Shannon
via quantization, MMD, centroid cosine distance).

The 14 datasets:

| # | Dataset | Role | Source |
|---|---|---|---|
| 1 | `synapsecai/synthetic-prompt-injections` | train | local (parquet) |
| 2 | `MPDD` | train | local (csv) |
| 3 | `TrustAIRLab/in-the-wild-jailbreak-prompts` | train | Hugging Face |
| 4 | `Harelix/Prompt-Injection-Mixed` | train | local (csv) |
| 5 | `jackhhao/jailbreak-classification` | train | Hugging Face |
| 6 | `qualifire/prompt-injections-benchmark` | train | Hugging Face |
| 7 | `jayavibhav/prompt-injection-safety` | train | Hugging Face (⚠️ `test` split only) |
| 8 | `toxic_scenarios` (ToxicDetector eval) | train | local (csv) |
| 10 | `nvidia/Aegis-AI-Content-Safety-Dataset-2.0` | train (**moved from OOD**) | Hugging Face (gated) |
| — | `hlyn-labs/prompt-injection-judge-deberta-dataset` | **OOD test** | Hugging Face |
| — | `allenai/wildjailbreak` (eval config) | **OOD test** | Hugging Face |
| — | LLM Past Tense (past-tense jailbreak reformulations) | **OOD test** | GitHub (tml-epfl/llm-past-tense) |
| — | `Necent/llm-jailbreak-prompt-injection-dataset` | **OOD test** | Hugging Face (filtered) |
| — | `TrustAIRLab/JailbreakQR` | **OOD test** | Hugging Face (snapshot download) |

**Caveats carried over / new:**
- **jayavibhav**: only the `test` split (10,000 rows) — matches what promptcop-7 was actually
  fine-tuned on.
- **toxic_scenarios**: seeded coin-flip between benign/malicious phrasing per row (reproducible
  stand-in for the original pipeline's unseeded draw).
- **Necent**: filtered to exclude rows whose `source` field matches any of your own training
  dataset names, and to English-only rows — this mirrors the filter already applied when this
  dataset was built into the held-out test set in the router notebook, so the distance check
  reflects the same rows actually being evaluated.


## Step 0 — Environment check (Python version + CUDA / RTX 3070)


In [ ]:
import sys, platform

EXPECTED_PYTHON = "3.11"   # <-- set to your pinned version

py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Running Python {py_version} ({sys.executable})")
print(f"Platform: {platform.platform()}")

if py_version != EXPECTED_PYTHON:
    print(f"\n⚠️  WARNING: expected Python {EXPECTED_PYTHON}, got {py_version}.")
else:
    print(f"\n✅ Python version matches EXPECTED_PYTHON ({EXPECTED_PYTHON}).")


In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {cuda_ok}")
if cuda_ok:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("No CUDA GPU detected — falling back to CPU (fine for this workload, just slower).")
    DEVICE = "cpu"
print(f"\nUsing device: {DEVICE}")


## Step 0b — Install dependencies


In [ ]:
%pip install -q sentence-transformers datasets scikit-learn scipy numpy matplotlib pandas pyarrow huggingface_hub requests


## Step 0c — Hugging Face login

Interactive prompt — token is **not** written into this notebook. Needed for the gated Aegis
dataset. Skip if you've already run `huggingface-cli login` or set `HF_TOKEN`.



In [ ]:
# from huggingface_hub import login

# login()


## Step 1 — Config


In [ ]:
from pathlib import Path
import numpy as np

LOCAL_DIR = Path("")   # <-- change: folder with the 4 local files
SAMPLE_SIZE = 2000           # prompts per dataset used for embedding/comparison
RANDOM_SEED = 42
OUTPUT_DIR = Path("./ood_distance_outputs_v2")
OUTPUT_DIR.mkdir(exist_ok=True)

for fname in ["synthetic-prompt-injections_train.parquet", "MPDD.csv",
              "harelix_data.csv", "toxic_scenarios (1).csv"]:
    p = LOCAL_DIR / fname
    print(f"  {'✅' if p.exists() else '❌ MISSING'}  {fname}")


## Step 2 — Loaders for the 4 local training datasets

Unchanged from the original notebook.


In [ ]:
import pandas as pd
import numpy as np

rng = np.random.RandomState(RANDOM_SEED)

def load_synthetic_prompt_injections():
    df = pd.read_parquet(LOCAL_DIR / "synthetic-prompt-injections_train.parquet")
    return df["text"].dropna().astype(str).tolist()

def load_mpdd():
    df = pd.read_csv(LOCAL_DIR / "MPDD.csv")
    df.columns = [c.strip() for c in df.columns]
    return df["Prompt"].dropna().astype(str).tolist()

def load_harelix():
    df = pd.read_csv(LOCAL_DIR / "harelix_data.csv", header=None, names=["prompt", "label"])
    return df["prompt"].dropna().astype(str).tolist()

def load_toxic_scenarios():
    df = pd.read_csv(LOCAL_DIR / "toxic_scenarios (1).csv")
    choices = rng.randint(0, 2, size=len(df))
    texts = np.where(choices == 0, df["Question_benign"], df["Question_malicious"])
    return pd.Series(texts).dropna().astype(str).tolist()

local_loaders = {
    "1_synthetic_prompt_injections": load_synthetic_prompt_injections,
    "2_MPDD": load_mpdd,
    "4_Harelix": load_harelix,
    "8_toxic_scenarios": load_toxic_scenarios,
}

local_datasets = {}
for name, loader in local_loaders.items():
    texts = loader()
    local_datasets[name] = texts
    print(f"{name}: {len(texts)} rows loaded")


## Step 3 — Loaders for the remaining training datasets (HF)

`guychuk` removed (dropped from training). `jayavibhav` still `test`-split-only. **Aegis is now
here, as training data**, not in the OOD section below.


In [ ]:
from datasets import load_dataset

def _col_guess(ds, candidates):
    """Return the first column name from `candidates` that exists in the dataset."""
    for c in candidates:
        if c in ds.column_names:
            return c
    raise KeyError(f"None of {candidates} found in columns {ds.column_names}")

def load_in_the_wild_jailbreak():
    ds = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", "jailbreak_2023_12_25", split="train")
    col = _col_guess(ds, ["prompt", "text"])
    return [str(x) for x in ds[col] if x]

def load_jackhhao():
    ds = load_dataset("jackhhao/jailbreak-classification", split="train")
    col = _col_guess(ds, ["prompt", "text"])
    return [str(x) for x in ds[col] if x]

def load_qualifire():
    ds = load_dataset("qualifire/prompt-injections-benchmark", split="test")
    col = _col_guess(ds, ["text", "prompt"])
    return [str(x) for x in ds[col] if x]

def load_jayavibhav():
    ds = load_dataset("jayavibhav/prompt-injection-safety", split="test")  # test-split-only, see notes
    col = _col_guess(ds, ["text", "prompt"])
    return [str(x) for x in ds[col] if x]

def load_aegis_train():
    # gated dataset — accept terms on the HF page + huggingface-cli login first
    ds = load_dataset("nvidia/Aegis-AI-Content-Safety-Dataset-2.0", split="train")
    col = _col_guess(ds, ["text", "prompt"])
    return [str(x) for x in ds[col] if x]

hf_train_loaders = {
    "3_in_the_wild_jailbreak": load_in_the_wild_jailbreak,
    "5_jackhhao": load_jackhhao,
    "6_qualifire": load_qualifire,
    "7_jayavibhav_TEST_SPLIT_ONLY": load_jayavibhav,
    "10_aegis_TRAIN": load_aegis_train,
}

hf_train_datasets = {}
for name, loader in hf_train_loaders.items():
    try:
        texts = loader()
        hf_train_datasets[name] = texts
        print(f"{name}: {len(texts)} rows loaded")
    except Exception as e:
        print(f"❌ {name}: FAILED — {type(e).__name__}: {e}")


## Step 4 — Loaders for the 5 new OOD test datasets

Reused verbatim from `bagel_router_v2_distilbert.ipynb`'s held-out test set construction, so
this distance check reflects the exact same rows actually used for evaluation, not a
re-sampled approximation of them.


In [ ]:
def load_hlyn():
    ds = load_dataset("hlyn-labs/prompt-injection-judge-deberta-dataset", split="train", token=True)
    ds = ds.shuffle(seed=RANDOM_SEED).select(range(min(2000, len(ds))))
    df = ds.to_pandas()
    return df["text"].dropna().astype(str).tolist()


def load_wildjailbreak():
    ds = load_dataset("allenai/wildjailbreak", "eval", delimiter="\t",
                      keep_default_na=False, split="train")
    df = ds.to_pandas()

    def get_prompt(row):
        if str(row["data_type"]).startswith("adversarial"):
            return row["adversarial"]
        return row["vanilla"]

    df["prompt"] = df.apply(get_prompt, axis=1)
    return df["prompt"].dropna().astype(str).tolist()


def load_past_tense():
    import requests
    api_url = ("https://api.github.com/repos/tml-epfl/llm-past-tense/"
               "contents/jailbreak_artifacts/past_tense")
    response = requests.get(api_url)
    response.raise_for_status()
    files = response.json()

    prompts = []
    for file_info in files:
        if file_info["name"].endswith(".json"):
            file_response = requests.get(file_info["download_url"])
            file_response.raise_for_status()
            data = file_response.json()
            for artifact in data["jb_artifacts"]:
                p = artifact.get("request_reformulated")
                if p is not None:
                    prompts.append(p)
    return list(dict.fromkeys(prompts))  # de-dup, preserve order


def load_necent():
    ds = load_dataset("Necent/llm-jailbreak-prompt-injection-dataset", split="train")

    training_source_patterns = [
        "synapsecai", "synthetic-prompt-injection", "mpdd", "malicious prompt detection",
        "trustairlab", "in-the-wild-jailbreak", "harelix", "prompt-injection-mixed-techniques",
        "jackhhao", "jailbreak-classification", "qualifire", "prompt-injections-benchmark",
        "jayavibhav", "prompt-injection-safety", "toxicdetector",
        "guychuk", "benign-malicious-prompt-classification",
        "aegis"
    ]
    multilingual_source_patterns = [
        "multijail", "ayaredteaming", "aya red teaming",
        "lumees", "multilingual-safety", "francophonia", "linguasafe",
    ]

    def allowed(source):
        s = str(source).lower()
        bad_training = any(p in s for p in training_source_patterns)
        bad_multilingual = any(p in s for p in multilingual_source_patterns)
        return not (bad_training or bad_multilingual)

    ds = ds.filter(lambda x: allowed(x["source"]))
    ds = ds.filter(lambda x: str(x["language"]).lower().startswith("en"))
    n = min(2000, len(ds))
    ds = ds.shuffle(seed=RANDOM_SEED).select(range(n))
    df = ds.to_pandas()
    return df["prompt"].dropna().astype(str).tolist()


def load_jailbreakqr():
    import os, json
    from huggingface_hub import snapshot_download

    repo_path = snapshot_download(repo_id="TrustAIRLab/JailbreakQR", repo_type="dataset", token=True)
    dataset_dir = os.path.join(repo_path, "dataset_json")

    prompts = []
    for filename in sorted(os.listdir(dataset_dir)):
        if filename.endswith(".json"):
            with open(os.path.join(dataset_dir, filename), "r", encoding="utf-8") as f:
                data = json.load(f)
            for item in data["jailbreaks"]:
                p = item.get("prompt")
                if p is not None and str(p).strip():
                    prompts.append(p)
    return prompts


ood_loaders = {
    "hlyn_OOD": load_hlyn,
    "wildjailbreak_OOD": load_wildjailbreak,
    "past_tense_OOD": load_past_tense,
    "necent_OOD": load_necent,
    "jailbreakqr_OOD": load_jailbreakqr,
}

ood_datasets = {}
for name, loader in ood_loaders.items():
    try:
        texts = loader()
        ood_datasets[name] = texts
        print(f"{name}: {len(texts)} rows loaded")
    except Exception as e:
        print(f"❌ {name}: FAILED — {type(e).__name__}: {e}")


## Step 5 — Combine, sample, and sanity-check the corpus


In [ ]:
all_datasets = {**local_datasets, **hf_train_datasets, **ood_datasets}

rng2 = np.random.RandomState(RANDOM_SEED)
datasets_sampled = {}
for name, texts in all_datasets.items():
    texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if SAMPLE_SIZE and len(texts) > SAMPLE_SIZE:
        idx = rng2.choice(len(texts), SAMPLE_SIZE, replace=False)
        texts = [texts[i] for i in idx]
    datasets_sampled[name] = texts

summary = pd.DataFrame([
    {"dataset": name, "n_used": len(texts), "mean_chars": np.mean([len(t) for t in texts]) if texts else 0}
    for name, texts in datasets_sampled.items()
]).sort_values("dataset").reset_index(drop=True)

print(summary.to_string(index=False))
print(f"\nTotal datasets loaded: {len(datasets_sampled)} / 14 (9 train + 5 OOD)")


## Step 6 — Embed every dataset into a shared vector space


In [ ]:
from sentence_transformers import SentenceTransformer
import time

EMBED_MODEL = "all-mpnet-base-v2"

model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"Loaded {EMBED_MODEL} on {DEVICE}")

embeddings = {}
for name, texts in datasets_sampled.items():
    t0 = time.time()
    emb = model.encode(texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True)
    embeddings[name] = emb
    print(f"{name}: embedded {len(texts)} prompts -> shape {emb.shape} ({time.time()-t0:.1f}s)")


## Step 7 — Distance / divergence metrics

Unchanged from the original notebook, including the conservative `pca_dims=15, eps=1e-3`
Gaussian-KL defaults (see prior discussion — reduces the chance of a near-singular covariance
inflating KL artificially, as happened with ahsanayub previously).


In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel
from scipy.spatial.distance import cosine, pdist
from scipy.stats import entropy


def centroid_cosine_matrix(embeddings):
    names = list(embeddings.keys())
    centroids = {k: v.mean(axis=0) for k, v in embeddings.items()}
    n = len(names)
    mat = np.zeros((n, n))
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            mat[i, j] = cosine(centroids[a], centroids[b])
    return names, mat


def gaussian_kl_matrix(embeddings, pca_dims=15, eps=1e-3):
    names = list(embeddings.keys())
    all_emb = np.vstack(list(embeddings.values()))
    pca_dims = min(pca_dims, all_emb.shape[0] - 1, all_emb.shape[1])
    pca = PCA(n_components=pca_dims, random_state=RANDOM_SEED).fit(all_emb)

    stats = {}
    for name, emb in embeddings.items():
        reduced = pca.transform(emb)
        mu = reduced.mean(axis=0)
        cov = np.cov(reduced, rowvar=False) + eps * np.eye(pca_dims)
        stats[name] = (mu, cov)

    def kl_gauss(mu_p, cov_p, mu_q, cov_q):
        k = mu_p.shape[0]
        cov_q_inv = np.linalg.inv(cov_q)
        diff = mu_q - mu_p
        term1 = np.trace(cov_q_inv @ cov_p)
        term2 = diff.T @ cov_q_inv @ diff
        sign_q, logdet_q = np.linalg.slogdet(cov_q)
        sign_p, logdet_p = np.linalg.slogdet(cov_p)
        term3 = logdet_q - logdet_p
        return 0.5 * (term1 + term2 - k + term3)

    n = len(names)
    mat = np.zeros((n, n))
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i != j:
                mat[i, j] = kl_gauss(*stats[a], *stats[b])
    return names, mat


def js_divergence_matrix(embeddings, n_clusters=50):
    names = list(embeddings.keys())
    all_emb = np.vstack(list(embeddings.values()))
    n_clusters = min(n_clusters, all_emb.shape[0] // 2)
    km = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10).fit(all_emb)

    hists = {}
    for name, emb in embeddings.items():
        labels = km.predict(emb)
        hist = np.bincount(labels, minlength=n_clusters).astype(float)
        hists[name] = hist / hist.sum()

    def js_div(p, q):
        m = 0.5 * (p + q)
        return 0.5 * entropy(p, m) + 0.5 * entropy(q, m)

    n = len(names)
    mat = np.zeros((n, n))
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            mat[i, j] = js_div(hists[a], hists[b])
    return names, mat


def mmd_matrix(embeddings, gamma=None):
    names = list(embeddings.keys())
    n = len(names)
    mat = np.zeros((n, n))

    if gamma is None:
        all_emb = np.vstack(list(embeddings.values()))
        sample = all_emb[rng2.choice(len(all_emb), min(500, len(all_emb)), replace=False)]
        median_dist = np.median(pdist(sample))
        gamma = 1.0 / (2 * median_dist ** 2 + 1e-8)

    def mmd(x, y):
        kxx = rbf_kernel(x, x, gamma=gamma).mean()
        kyy = rbf_kernel(y, y, gamma=gamma).mean()
        kxy = rbf_kernel(x, y, gamma=gamma).mean()
        return kxx + kyy - 2 * kxy

    for i, a in enumerate(names):
        for j, b in enumerate(names):
            mat[i, j] = mmd(embeddings[a], embeddings[b])
    return names, mat

print("metric functions ready")


## Step 8 — Run all four metrics and plot heatmaps


In [ ]:
import matplotlib.pyplot as plt

def plot_heatmap(mat, names, title, out_path):
    fig, ax = plt.subplots(figsize=(max(7, len(names)*0.7), max(6, len(names)*0.7)))
    im = ax.imshow(mat, cmap="viridis")
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=8)
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.show()

metric_fns = {
    "cosine": centroid_cosine_matrix,
    "gaussian_kl": gaussian_kl_matrix,
    "js_divergence": js_divergence_matrix,
    "mmd": mmd_matrix,
}

results = {}
for metric_name, fn in metric_fns.items():
    names, mat = fn(embeddings)
    results[metric_name] = (names, mat)
    pd.DataFrame(mat, index=names, columns=names).to_csv(OUTPUT_DIR / f"{metric_name}_matrix.csv")
    plot_heatmap(mat, names, metric_name, OUTPUT_DIR / f"{metric_name}_heatmap.png")
    print(f"Saved {metric_name} matrix + heatmap to {OUTPUT_DIR}/")


## Step 9 — Nearest-training-dataset check per OOD test set


In [ ]:
ood_names = [n for n in embeddings if n.endswith("_OOD")]
train_names = [n for n in embeddings if not n.endswith("_OOD")]

print(f"OOD test datasets found: {ood_names}")
print(f"Training datasets found: {train_names}\n")

for metric_name, (names, mat) in results.items():
    df_m = pd.DataFrame(mat, index=names, columns=names)
    print(f"=== {metric_name} — nearest training dataset per OOD test set ===")
    for ood in ood_names:
        if ood not in df_m.index:
            continue
        row = df_m.loc[ood, train_names]
        nearest = row.idxmin()
        print(f"  {ood:20s} -> closest: {nearest:35s} ({metric_name}={row[nearest]:.4f})   "
              f"[mean to train pool: {row.mean():.4f}]")
    print()


## Notes

- `guychuk` and `ahsanayub` are intentionally absent from this notebook — dropped from
  training and OOD respectively, per the current pipeline composition.
- If Step 10 turns up any hits, treat it the same way as the ahsanayub/Harelix finding: don't
  just patch the offending rows out of the OOD set quietly — flag it, since it likely means a
  broader "is this really disjoint" question needs asking about wherever that overlap came from.
- Same `SAMPLE_SIZE` tradeoff as before: 1000/dataset for iteration speed; bump up for final
  numbers once everything loads cleanly.
